In [1]:
#program 1
import cv2
import numpy as np
 
cropping = False
x_start, y_start, x_end, y_end = 0, 0, 0, 0
 
image = cv2.imread('template.jpg')
image2 = image.copy()
 
def mouse_crop(event, x, y, flags, param):
    global x_start, y_start, x_end, y_end, cropping
 
    if event == cv2.EVENT_LBUTTONDOWN:
        x_start, y_start, x_end, y_end = x, y, x, y
        cropping = True
    elif event == cv2.EVENT_MOUSEMOVE:
        if cropping == True:
            x_end, y_end = x, y
    elif event == cv2.EVENT_LBUTTONUP:
        x_end, y_end = x, y
        cropping = False 
        refPoint = [(x_start, y_start), (x_end, y_end)] 
        if len(refPoint) == 2:
            roi = image2[refPoint[0][1]:refPoint[1][1], refPoint[0][0]:refPoint[1][0]]
            cv2.imshow("roi image", roi)
            cv2.imwrite("stopsign_roi.jpg", roi)
            
cv2.namedWindow("image")
cv2.setMouseCallback("image", mouse_crop)
 
while True: 
    i = image.copy() 
    if not cropping:
        cv2.imshow("image", image) 
    elif cropping:
        cv2.rectangle(i, (x_start, y_start), (x_end, y_end), (0, 0, 255), 2)
        cv2.imshow("image", i)
 
    ch = cv2.waitKey(1)
    if ch == ord('q'):
        break
    
cv2.destroyAllWindows()

In [3]:
import cv2

# -----------------------------
# Load template
# -----------------------------
template = cv2.imread("template_roi.jpg")

if template is None:
    print("Error: template.jpg not found")
    exit()

template_gray = cv2.cvtColor(template, cv2.COLOR_BGR2GRAY)

# Template dimensions
h, w = template_gray.shape

# -----------------------------
# Camera
# -----------------------------
cap = cv2.VideoCapture(1)

# -----------------------------
# Video recording
# -----------------------------
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(
    "template_matching_result.mp4",
    fourcc,
    20.0,
    (640, 480)
)

# Matching threshold
threshold = 0.75

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Resize camera frame
    frame = cv2.resize(frame, (640, 480))

    # Convert frame to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Template matching
    result = cv2.matchTemplate(
        gray,
        template_gray,
        cv2.TM_CCOEFF_NORMED
    )

    # Find best match
    min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)

    # -----------------------------
    # Check whether object is found
    # -----------------------------
    if max_val >= threshold:

        x, y = max_loc

        # Draw bounding box
        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        # Display Found
        cv2.putText(
            frame,
            f"Found ({max_val:.2f})",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

    else:

        # Display Not Found
        cv2.putText(
            frame,
            f"Not Found ({max_val:.2f})",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255),
            2
        )

    # Show result
    cv2.imshow("Template Matching", frame)

    # Save frame to video
    out.write(frame)

    # Press q to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# -----------------------------
# Release everything
# -----------------------------
cap.release()
out.release()
cv2.destroyAllWindows()

print("Video saved as template_matching_result.mp4")

Video saved as template_matching_result.mp4
